# Once Upon a Mirror, notebook 1: The estimate

In section 1 of the game, the mirror could not see inside Cinderella's urns. It saw only the dots of the board,
🟠 for a tap and ⚪ for no tap, and its estimate of a product was p̂ = orange dots / dots. Now you teach that to
a computer. A tap is written 1 and no tap 0, so each row of the board becomes a list of 1s and 0s, and from these
lists you compute the estimate of each product.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. It takes about 20 minutes.

The products, the customers and their probabilities are invented for the game. Start with the two folded cells
below: the mirror's kit, then the checks.

In [ ]:
#@title The mirror's kit (run me first) { display-mode: "form" }
# The mirror's kit. Run this cell first; you do not need to read it.
SEASON = 100            # mornings in a season
RACE = 103993           # the seed of the race season: the season of every comparison in the game
NAMES = ['Robot vacuum', 'Rope ladder', 'Leaf blower', 'Memory-foam mattress', 'Satnav']
EMOJI = ['🤖', '🪜', '🍃', '🛏️', '🧭']
_KIT_STREAMS = {"random": 1, "estimate then exploit": 2, "epsilon-greedy": 3, "ucb": 4}

def random_numbers(seed):
    """A stream of random numbers between 0 and 1: next_number = random_numbers(7), then next_number() gives the
    first number, next_number() again the second, and so on. The same seed gives the same numbers as in the game."""
    state = seed & 0xFFFFFFFF

    def next_number():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xFFFFFFFF
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xFFFFFFFF
        t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
        return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296

    return next_number


def strategy_numbers(seed, strategy):
    """A strategy's own stream of random numbers in the season with this seed, as in the game. strategy is one of
    "random", "estimate then exploit", "epsilon-greedy" and "ucb"."""
    return random_numbers(10 * seed + _KIT_STREAMS[strategy])


def season_moods(seed):
    """The customer's mood on each morning of the season with this seed: SEASON numbers between 0 and 1."""
    next_number = random_numbers(seed)
    moods = []
    for _ in range(SEASON):
        moods.append(next_number())
    return moods


def tap(p, mood):
    """1 if the customer taps the product shown, else 0. She taps when her mood is below p, which happens on a
    share p of the mornings. Every mirror of a season meets the same moods, so the comparison is fair."""
    return 1 if mood < p else 0


def _kit_is_board(rows):
    if not isinstance(rows, (list, tuple)) or len(rows) == 0:
        return False
    for row in rows:
        if not isinstance(row, (list, tuple)):
            return False
        for x in row:
            if x not in (0, 1):
                return False
    return True


def _kit_number(x):
    """At most two decimals, as in the game: 0.6, 0.67, 1, 0."""
    text = "%.2f" % x
    return text.rstrip("0").rstrip(".") if "." in text else text


def rate(rows):
    """The rate of a season: taps / mornings over the whole board."""
    if not _kit_is_board(rows):
        print("⏳ rate needs a board: one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return None
    taps = sum(sum(row) for row in rows)
    mornings = sum(len(row) for row in rows)
    return taps / mornings if mornings else 0


def _kit_short(x):
    text = repr(x)
    return text if len(text) <= 60 else text[:57] + "..."


def show_board(rows, symbols=None, highlight=None, width=30, estimates=True):
    """Print the board as in the game: the estimate p-hat, the product, then one dot per showing, 🟠 a tap and
    ⚪ no tap. highlight: a row (or a list of rows) to mark with ⭐. A row longer than width is cut and its
    remaining dots are counted."""
    if not _kit_is_board(rows):
        print("⏳ No board to show yet: show_board needs one list of 1s and 0s per product. It got %s." % _kit_short(rows))
        return
    symbols = symbols or EMOJI
    marked = highlight if isinstance(highlight, (list, tuple)) else ([] if highlight is None else [highlight])
    if estimates:
        print("    p\u0302")
    for k in range(len(rows)):
        row = rows[k]
        est = ("?" if not row else _kit_number(sum(row) / len(row))) if estimates else ""
        dots = "".join("🟠" if x else "⚪" for x in row[:width])
        if len(row) > width:
            rest = row[width:]
            dots += " ... and %d more (%d 🟠)" % (len(rest), sum(rest))
        name = symbols[k] if k < len(symbols) else "#%d" % k
        print("%5s  %s  %s%s" % (est, name, dots, "  ⭐" if k in marked else ""))

In [ ]:
#@title The checks of this notebook (run me too) { display-mode: "form" }
# The checks of notebook 1, and show_rounds. Run this cell after the kit; you do not need to read it.
# A check prints one short message and never stops the notebook.

_n1_NAMES = ['Robot vacuum', 'Rope ladder', 'Leaf blower', 'Memory-foam mattress', 'Satnav']
_n1_EMOJI = ['🤖', '🪜', '🍃', '🛏️', '🧭']

# Each test: the inputs, the expected answer, the call as you would type it, and why that is the answer
_n1_TESTS = {
    "estimate": [
        (([0, 1, 0, 1, 1],), 0.6,
         'estimate([0, 1, 0, 1, 1])',
         "The vacuum's first 5 showings in the race season: 3 taps in 5 showings"),
        (([1, 0, 1, 0, 0],), 0.4,
         'estimate([1, 0, 1, 0, 0])',
         'The mattress: 2 taps in 5 showings'),
        (([1],), 1.0,
         'estimate([1])',
         'The mattress after its first showing: 1 tap in 1 showing'),
        (([0, 0, 0, 0, 0],), 0.0,
         'estimate([0, 0, 0, 0, 0])',
         'The ladder: no tap in 5 showings'),
        (([],), None,
         'estimate([])',
         'A product never shown has no estimate yet: None, where the game writes ?'),
        (([0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0],), 0.6,
         "estimate on the vacuum's row of a season of the Queen's mirror",
         'It shows the vacuum every morning: 60 taps in 100 showings'),
        (([0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1],), 0.8,
         "estimate on the ladder's row of a season at Rapunzel's",
         '12 taps in 15 showings'),
        (([1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0],), 0.47619047619047616,
         "estimate on the mattress's row of a season at Cinderella's",
         '10 taps in 21 showings, which is 0.47619047619047616 with every decimal'),
    ],
    "estimates": [
        (([[0, 1, 0, 1, 1], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [1, 0, 1, 0, 0], [0, 0, 0, 0, 0]],), [0.6, 0.0, 0.0, 0.4, 0.0],
         'estimates(race_board)',
         'The board of the first 25 mornings: its p̂ column'),
        (([[0], [0], [0], [], []],), [0.0, 0.0, 0.0, None, None],
         'estimates([[0], [0], [0], [], []])',
         'The board after 3 mornings: the mattress and the satnav were never shown yet, so their estimates are None'),
        (([[0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0], [], [], [], []],), [0.6, None, None, None, None],
         "estimates on a season of the Queen's mirror at Cinderella's",
         'It showed the vacuum on all 100 mornings and never the other four products: None for them'),
        (([[0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0]],), [0.4666666666666667, 0.06666666666666667, 0.07142857142857142, 0.47619047619047616, 0.25],
         "estimates on a season at Cinderella's in which every product was shown",
         'One estimate per row, such as 14 taps in 30 showings for the vacuum'),
        (([[0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0]],), [0.3, 0.8, 0.07142857142857142, 0.23809523809523808, 0.25],
         "estimates on a season at Rapunzel's in which every product was shown",
         'One estimate per row, such as 12 taps in 15 showings for the ladder'),
        (([[1, 0, 1, 0, 0]],), [0.4],
         'estimates([[1, 0, 1, 0, 0]])',
         "A board with one row, the mattress's: a list that holds one estimate"),
    ],
    "leaders": [
        (([0.0, 0.0, 0.0, 1.0, 0.0],), [3],
         'leaders([0, 0, 0, 1, 0])',
         'After round 1: the mattress leads alone, 1 tap in 1 showing'),
        (([0.5, 0.0, 0.0, 0.5, 0.0],), [0, 3],
         'leaders([0.5, 0, 0, 0.5, 0])',
         'After round 2: the vacuum and the mattress share the highest estimate, 0.5'),
        (([0.3333333333333333, 0.0, 0.0, 0.6666666666666666, 0.0],), [3],
         'leaders([0.33, 0, 0, 0.67, 0])',
         'After round 3: the mattress alone, 0.67'),
        (([0.5, 0.0, 0.0, 0.5, 0.0],), [0, 3],
         'leaders([0.5, 0, 0, 0.5, 0])',
         'After round 4: the vacuum and the mattress tie again, 0.5'),
        (([0.6, 0.0, 0.0, 0.4, 0.0],), [0],
         'leaders([0.6, 0, 0, 0.4, 0])',
         'After round 5: the vacuum alone, 0.6 against 0.4'),
        (([0.4666666666666667, 0.06666666666666667, 0.07142857142857142, 0.47619047619047616, 0.25],), [3],
         'leaders([0.47, 0.07, 0.07, 0.48, 0.25])',
         "The estimates of a season at Cinderella's: the mattress, 0.48, just above the vacuum, 0.47"),
        (([0.0, 0.0, 0.0],), [0, 1, 2],
         'leaders([0, 0, 0])',
         'Three products with no tap, all at 0: all three rows share the highest value'),
        (([0.6, 0.2, 0.2, 0.4, 0.2],), [0],
         'leaders([0.6, 0.2, 0.2, 0.4, 0.2])',
         "Cinderella's probabilities p: the vacuum, row 0, the product the Queen's mirror shows her"),
        (([0.4, 0.8, 0.2, 0.2, 0.2],), [1],
         'leaders([0.4, 0.8, 0.2, 0.2, 0.2])',
         "Rapunzel's probabilities p: the ladder, row 1"),
    ],
}

_n1_SUCCESS = {'estimate': "The vacuum's 3 taps in 5 showings give 0.6, its 60 taps in 100 showings in a season of the Queen's mirror give 0.6 as well, and a product never shown gives None.", 'estimates': 'For race_board it gives [0.6, 0, 0, 0.4, 0], the p̂ column of the board, and None for every product that a board never showed.', 'leaders': "Round by round on race_board it finds [3], [0, 3], [3], [0, 3] and [0]. On Cinderella's probabilities it finds [0], the vacuum, the product the Queen's mirror shows her."}

# What the usual slips of estimate and of leaders give on their tests ("skip" for the empty row, "error" for a
# crash), each with its hint
_n1_SLIPS = [
    ('ones', [1.0, 1.0, 1.0, 1.0, 'skip', 1.0, 1.0, 1.0],
     'taps = taps + 1 counts every dot, the no taps too. Add the dot itself, taps = taps + x: it adds 1 for a tap and 0 for no tap.'),
    ('self', [1.0, 1.0, 1.0, 'error', 'skip', 1.0, 1.0, 1.0],
     'showings is the number of dots, len(row): every showing counts, with a tap or without one.'),
    ('count', [3, 2, 1, 0, 'skip', 60, 12, 10],
     'your function gives the number of taps. The estimate divides it by the number of showings: return taps / showings.'),
    ('showings', [5, 5, 1, 5, 'skip', 100, 15, 21],
     'your function gives the number of showings. The estimate is the taps divided by it: taps / showings.'),
    ('upside', [1.6666666666666667, 2.5, 1.0, 'error', 'skip', 1.6666666666666667, 1.25, 2.1],
     'the division is upside down: the taps go on top, taps / showings.'),
    ('floor', [0, 0, 1, 0, 'skip', 0, 0, 0],
     '// drops the decimals: 3 // 5 is 0. Divide with a single /: taps / showings.'),
    ('percent', [60.0, 40.0, 100.0, 0.0, 'skip', 60.0, 80.0, 47.61904761904762],
     'the estimate is a share of the showings, a number from 0 to 1: 3 taps in 5 showings give 0.6. Leave out the 100.'),
    ('notaps', [0.4, 0.6, 0.0, 1.0, 'skip', 0.4, 0.2, 0.5238095238095238],
     'this is the share of the no taps. Count the taps: x is 1 for a tap, so taps = taps + x.'),
    ('last', [0.2, 0.0, 1.0, 0.0, 'skip', 0.0, 0.06666666666666667, 0.0],
     'taps = x forgets the taps counted before. Add x to them: taps = taps + x.'),
    ('first', [0.0, 0.2, 1.0, 0.0, 'skip', 0.0, 0.0, 0.047619047619047616],
     'the function stops after the first dot, so its last lines run inside the loop. Write the line of showings and the return line after the loop, at the same indentation as for.'),
]
_n1_LEADER_SLIPS = [
    ('lowest', [[0, 1, 2, 4], [1, 2, 4], [1, 2, 4], [1, 2, 4], [1, 2, 4], [1], [0, 1, 2], [1, 2, 4], [2, 3, 4]],
     'these are the rows with the lowest value. In the first loop, highest takes the value when value > highest.'),
    ('last', [[0, 1, 2, 4], [1, 2, 4], [1, 2, 4], [1, 2, 4], [1, 2, 4], [4], [0, 1, 2], [1, 2, 4], [2, 3, 4]],
     'these are the rows whose value equals the last value of the list. The second if compares the value of row k with highest: values[k] == highest.'),
    ('first', [[0, 1, 2, 4], [0, 3], [0], [0, 3], [0], [0], [0, 1, 2], [0], [0]],
     'these are the rows whose value equals the first value. The first loop must end with the highest value: highest takes the value whenever value > highest.'),
    ('values', [[1.0], [0.5, 0.5], [0.6666666666666666], [0.5, 0.5], [0.6], [0.47619047619047616], [0.0, 0.0, 0.0], [0.6], [0.8]],
     'best_rows holds the values of the rows. Append k, the number of the row.'),
    ('shifted', [[4], [1, 4], [4], [1, 4], [1], [4], [1, 2, 3], [1], [2]],
     'Python counts the rows from 0: the vacuum is row 0, the ladder row 1, and so on.'),
    ('all', [[0, 1, 2, 3, 4], [0, 1, 2, 3, 4], [0, 1, 2, 3, 4], [0, 1, 2, 3, 4], [0, 1, 2, 3, 4], [0, 1, 2, 3, 4], [0, 1, 2], [0, 1, 2, 3, 4], [0, 1, 2, 3, 4]],
     'every row went into the list. The second if asks whether the value of row k equals the highest: values[k] == highest.'),
    ('empty', [[], [], [], [], [], [], [], [], []],
     'best_rows stays empty. In the second loop, a row goes into it when its value equals the highest: values[k] == highest, with two equal signs.'),
]


def _n1_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = repr(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 70:
        text = text[:67] + "..."
    return text


def _n1_is_number(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def _n1_number(x):
    """A number as the messages write it: 0.6, 0.67, 1, 0, and four decimals when two would hide a difference."""
    try:
        if abs(x - round(x, 2)) < 1e-12:
            text = "%.2f" % x
        else:
            text = "%.4f" % x
    except Exception:
        return _n1_short(x)
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    return text


def _n1_dec(x):
    """At most two decimals, as on the board: 0.6, 0.67, 1, 0."""
    text = "%.2f" % x
    if "." in text:
        text = text.rstrip("0").rstrip(".")
    return text


def _n1_text(value):
    """A value as the messages write it: a number with few decimals, a short list of them, None."""
    if _n1_is_number(value):
        return _n1_number(value)
    if isinstance(value, list) and len(value) <= 12:
        parts = []
        for x in value:
            parts.append(_n1_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n1_short(value)


def _n1_and(items):
    """a, b and c"""
    items = [str(x) for x in items]
    if len(items) == 1:
        return items[0]
    return ", ".join(items[:-1]) + " and " + items[-1]


def _n1_label(k, rows):
    """The name of row k of a board with this many rows: 🤖 Robot vacuum on a board of the Mirror Shop, else row k."""
    if rows == len(_n1_NAMES) and isinstance(k, int) and 0 <= k < rows:
        return "%s %s" % (_n1_EMOJI[k], _n1_NAMES[k])
    return "row %s" % _n1_short(k)


def _n1_same(answer, expected):
    """True when answer is the expected value: numbers up to 1e-9, None only for None, lists entry by entry."""
    if expected is None:
        return answer is None
    if _n1_is_number(expected):
        return _n1_is_number(answer) and abs(answer - expected) <= 1e-9
    if isinstance(expected, list):
        if not isinstance(answer, list) or len(answer) != len(expected):
            return False
        for i in range(len(expected)):
            if not _n1_same(answer[i], expected[i]):
                return False
        return True
    return answer == expected


def _n1_call(fn, inputs):
    """Run a participant's function on fresh copies of the inputs, with its prints silenced.
    Gives back (answer, error); error is None when the call went through. Never raises."""
    import contextlib, copy, io
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            answer = fn(*copy.deepcopy(inputs))
        return answer, None
    except Exception as error:
        return None, "%s: %s" % (type(error).__name__, error)


def _n1_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    # A gap such as "x = ..." stays in the compiled function; a gap that stands on its own line is only visible in
    # the source, which Jupyter and Colab keep for every cell
    code = getattr(fn, "__code__", None)
    if code is not None and Ellipsis in code.co_consts:
        return True
    import inspect
    try:
        lines = inspect.getsource(fn).splitlines()
    except Exception:
        return False
    for line in lines:
        if "..." in line.split("#")[0]:
            return True
    return False


def _n1_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n1_first_failure(name, fn):
    """The first test of name that fn fails, or None when fn passes them all."""
    for inputs, expected, call, why in _n1_TESTS[name]:
        answer, error = _n1_call(fn, inputs)
        if error is not None or not _n1_same(answer, expected):
            return {"inputs": inputs, "answer": answer, "error": error, "expected": expected, "call": call,
                    "why": why}
    return None


def _n1_works(name):
    """True when the function called name exists, is finished and passes its tests."""
    fn = globals().get(name)
    if not callable(fn) or _n1_has_gap(fn):
        return False
    return _n1_first_failure(name, fn) is None


# The hints

def _n1_matches(answers, values):
    """True when the answers on all the tests are those of one slip."""
    for i in range(len(values)):
        wanted, got = values[i], answers[i]
        if isinstance(wanted, str):
            if wanted == "skip":
                continue
            if not (isinstance(got, str) and got == "error"):
                return False
        elif not _n1_is_number(got) or abs(got - wanted) > 1e-9:
            return False
    return True


def _n1_hint_estimate(fn, failure):
    row = failure["inputs"][0]
    answer, error, expected = failure["answer"], failure["error"], failure["expected"]
    if len(row) == 0:
        if error is not None:
            return ("an empty row has no showings, and dividing by 0 is an error. Keep the first two lines of the "
                    "function: for an empty row they return None, before any division.")
        return ("a product never shown has no estimate yet. Keep the first two lines of the function: for an empty "
                "row they return None.")
    answers = []
    for inputs, wanted, call, why in _n1_TESTS["estimate"]:
        got, problem = _n1_call(fn, inputs)
        if problem is not None:
            got = "error"
        answers.append(got)
    for name, values, hint in _n1_SLIPS:
        if _n1_matches(answers, values):
            return hint
    if error is not None:
        return ("count the taps in the loop (taps = taps + x), the showings with len(row), then divide: "
                "taps / showings.")
    if answer is None:
        return "end the function with a return line: return taps / showings."
    if not _n1_is_number(answer):
        return "return one number, the result of taps / showings."
    for places in (1, 2, 3):
        if abs(answer - round(expected, places)) < 1e-12:
            return ("keep every decimal: return taps / showings as it comes, without round. show_board rounds the "
                    "numbers it prints.")
    return "count the taps in the loop (taps = taps + x), the showings with len(row), then return taps / showings."


def _n1_hint_estimates(fn, failure):
    rows = failure["inputs"][0]
    answer, error, expected = failure["answer"], failure["error"], failure["expected"]
    if error is not None:
        if "ZeroDivisionError" in error:
            return ("a product that was never shown reached a division. Your estimate from task A returns None for "
                    "it: value = estimate(row).")
        if "TypeError" in error:
            return "estimate takes one row, a list of 1s and 0s: estimate(row). rows is the whole board."
        return "in the loop, value = estimate(row), then values.append(value)."
    if answer is None:
        return "end the function with return values, after the loop."
    if not isinstance(answer, list):
        return "return the list values, with one estimate for each row of the board."
    if len(answer) == 0:
        return ("values is still empty: inside the loop, put each value at the end of the list with "
                "values.append(value).")
    for x in answer:
        if isinstance(x, (list, tuple)):
            return ("each entry of values is one number, an estimate. Did you append the row itself? Append value, "
                    "its estimate.")
    if len(answer) == 1 and len(expected) > 1:
        return ("the function stops after the first row: the line return values belongs after the loop, at the "
                "same indentation as for.")
    if len(answer) != len(expected):
        return "one estimate per row: append exactly once in each round of the loop."
    if len(answer) > 1 and answer.count(answer[0]) == len(answer) and expected.count(expected[0]) != len(expected):
        return "every estimate is the same: in each round of the loop, use the row of that round, estimate(row)."
    for k in range(len(expected)):
        if not _n1_same(answer[k], expected[k]):
            label = _n1_label(k, len(rows))
            if expected[k] is None:
                return ("%s was never shown, so its estimate is None. Append what estimate(row) gives, None "
                        "included." % label)
            return ("%s should get %s and got %s. In each round of the loop, value = estimate(row)."
                    % (label, _n1_text(expected[k]), _n1_text(answer[k])))
    return "in the loop, value = estimate(row), then values.append(value)."


def _n1_hint_leaders(fn, failure):
    values = failure["inputs"][0]
    answer, error, expected = failure["answer"], failure["error"], failure["expected"]
    if error is not None:
        if "IndexError" in error:
            return "k counts the rows from 0, and values[k] is the value of row k."
        if "TypeError" in error:
            return ("each if compares two numbers: value > highest in the first loop, values[k] == highest in the "
                    "second.")
        return "first loop: if value > highest. Second loop: if values[k] == highest."
    if answer is None:
        return "end the function with return best_rows, after the second loop."
    if _n1_is_number(answer):
        return "that is the highest value itself. Return best_rows, the list of the rows that have it."
    if not isinstance(answer, list):
        return "return best_rows, the list of the numbers of the rows."
    answers = []
    for inputs, wanted, call, why in _n1_TESTS["leaders"]:
        got, problem = _n1_call(fn, inputs)
        if problem is not None:
            got = "error"
        answers.append(got)
    for name, vector, hint in _n1_LEADER_SLIPS:
        if answers == vector:
            return hint
    if len(answer) == 0:
        return ("best_rows stays empty here. In the second loop, a row goes into it when its value equals the "
                "highest: values[k] == highest, with two equal signs.")
    top = max(values)
    inside = True
    for k in answer:
        if k not in expected:
            inside = False
    if len(expected) > 1 and len(answer) < len(expected) and inside:
        names = []
        for k in expected:
            names.append(_n1_label(k, len(values)))
        return ("%s share the highest value, %s, so each of them belongs in the list. Collect every k with "
                "values[k] == highest." % (_n1_and(names), _n1_number(top)))
    return ("the highest value is %s, and the rows that have it are %s. Collect every k with values[k] == highest."
            % (_n1_number(top), _n1_text(expected)))


_n1_NEEDS = {"estimate": [], "estimates": [("estimate", "task A")], "leaders": []}
_n1_HINTS = {"estimate": _n1_hint_estimate, "estimates": _n1_hint_estimates, "leaders": _n1_hint_leaders}


def _n1_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    not_finished = ("⏳ %s is not finished yet: replace each ... with your code, run its cell, then run this check "
                    "again." % name)
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _n1_has_gap(fn):
        print(not_finished)
        return
    for other, place in _n1_NEEDS[name]:
        if _n1_uses(fn, other) and not _n1_works(other):
            print("⏳ %s uses your %s from %s, which does not work yet. Get check_%s(%s) to ✅ first, then run this "
                  "check again." % (name, other, place, other, other))
            return
    failure = _n1_first_failure(name, fn)
    if failure is None:
        print("✅ %s works on %d tests. %s" % (name, len(_n1_TESTS[name]), _n1_SUCCESS[name]))
        return
    if failure["answer"] is Ellipsis or "ellipsis" in str(failure["error"]).lower():
        print(not_finished)
        return
    hint = _n1_HINTS[name](fn, failure)
    if failure["error"] is not None:
        print("❌ %s stopped with an error: %s" % (failure["call"], failure["error"]))
    elif failure["answer"] is None:
        print("❌ %s gave no answer (None), expected %s. %s." % (failure["call"], _n1_text(failure["expected"]),
                                                                 failure["why"]))
    else:
        print("❌ %s gave %s, expected %s. %s." % (failure["call"], _n1_text(failure["answer"]),
                                                   _n1_text(failure["expected"]), failure["why"]))
    print("   💡 Hint: " + hint)


def check_estimate(fn):
    """Task A: tests estimate on eight rows, from an empty row to a whole season."""
    try:
        _n1_check("estimate", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two folded cells at the top again, then this cell." % error)


def check_estimates(fn):
    """Task B: tests estimates on six boards, from one row to whole seasons of both customers."""
    try:
        _n1_check("estimates", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two folded cells at the top again, then this cell." % error)


def check_leaders(fn):
    """Task C: tests leaders on the estimates of the five rounds and on four more lists."""
    try:
        _n1_check("leaders", fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the two folded cells at the top again, then this cell." % error)


# show_rounds: the board after each round, with a star on the leaders that the participant's functions find

def _n1_is_board(rows):
    if not isinstance(rows, (list, tuple)) or len(rows) == 0:
        return False
    for row in rows:
        if not isinstance(row, (list, tuple)):
            return False
        for x in row:
            if x not in (0, 1):
                return False
    return True


def _n1_round_leaders(board):
    """The leaders of a board found by the participant's estimates and leaders, or None when they fail on it."""
    values, problem = _n1_call(globals()["estimates"], (board,))
    if problem is not None or not isinstance(values, list) or len(values) != len(board):
        return None, None
    best, problem = _n1_call(globals()["leaders"], (values,))
    if problem is not None or not isinstance(best, list) or len(best) == 0:
        return None, None
    for k in best:
        if not isinstance(k, int) or k < 0 or k >= len(board):
            return None, None
    return best, values[best[0]]


def show_rounds(rows):
    """Print the board rows after each round, the first dot of every row, then the first two, and so on, with a ⭐
    on the leaders that your estimates (task B) and leaders (task C) find. Until both work, the stars are missing."""
    try:
        if not _n1_is_board(rows):
            print("⏳ show_rounds needs a board: one list of 1s and 0s per product. It got %s." % _n1_short(rows))
            return
        rounds = len(rows[0])
        for row in rows:
            rounds = min(rounds, len(row))
        if rounds == 0:
            print("⏳ A row of this board is still empty, so the board has no whole round yet.")
            return
        ready = _n1_works("estimates") and _n1_works("leaders")
        if not ready:
            print("⏳ The ⭐ will mark the leaders once your estimates (task B) and leaders (task C) pass their "
                  "checks. Until then, here are the boards without them.")
            print()
        for r in range(1, rounds + 1):
            board = []
            for row in rows:
                board.append(list(row[:r]))
            line = "After round %d (%d mornings)" % (r, r * len(rows))
            best = None
            if ready:
                best, top = _n1_round_leaders(board)
            if best is None:
                print(line + ":")
            else:
                names = []
                for k in best:
                    names.append(_n1_label(k, len(rows)))
                if len(best) == 1:
                    print("%s: %s leads with %s." % (line, names[0], _n1_dec(top)))
                else:
                    print("%s: %s share the lead with %s." % (line, _n1_and(names), _n1_dec(top)))
            show_board(board, highlight=best)
            print()
    except Exception as error:
        print("❌ show_rounds could not run (%s). Run the two folded cells at the top again, then this cell." % error)


print("✅ The checks of notebook 1 are ready.")

## One product: a row of 1s and 0s

Each morning the mirror shows Cinderella one product, and she taps it or not. On the board, every showing is a
dot in the product's row. In Python the row is a list, with 1 for a tap and 0 for no tap, in the order of the
showings.

The game plays one season in every comparison, the **race season**. Here are the mattress's first 5 showings
in it. `show_board` draws the row as the game does, with the estimate p̂ on its left.

In [ ]:
# The mattress at Cinderella's: its first 5 showings in the race season, 1 for a tap and 0 for no tap
mattress = [1, 0, 1, 0, 0]
show_board([mattress], symbols=["🛏️"])

**Reading the result.** 2 orange dots out of 5, so p̂ = 2 / 5 = 0.4. With 1s and 0s: (1 + 0 + 1 + 0 + 0) / 5 = 2 / 5 = 0.4.
Adding up the 1s and 0s counts the taps, so the estimate is the **average** of the row: taps / showings.
(`symbols=["🛏️"]` gives this board of one row its picture; a board of the five products gets theirs by itself.)

## 🎯 Task A. The estimate of one product

Write `estimate(row)`. It gets the row of one product, a list of 1s and 0s, and returns its estimate,
taps / showings. `estimate(mattress)` should give 0.4.

A product that was never shown has an empty row, `[]`, and no estimate yet: the game writes "?", and Python has
`None`. The first two lines of the function take care of that. You fill in the rest, in three small steps:

1. **Count the taps.** The loop `for x in row` visits the dots one by one, and `x` is 1 for a tap and 0 for no
   tap. `taps` starts at 0: add each `x` to it.
2. **Count the showings**: one per dot, so the length of the row, `len(row)`.
3. **Divide**: taps / showings.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# Adding up the dots of a row counts its taps: for the mattress, 1 + 0 + 1 + 0 + 0 = 2, its 2 taps.
# So the loop adds each x to taps: a tap adds 1, no tap adds 0.

In [ ]:
#@title 💡 Hint 2 for task A (double-click to open) { display-mode: "form" }
# taps = taps + x takes the taps counted so far and adds this dot.
# After the loop, len(row) is the number of dots, which is the number of showings.

In [ ]:
#@title 💡 Hint 3 for task A: the shape of the code (double-click to open) { display-mode: "form" }
#     taps = 0
#     for x in row:
#         taps = taps + x
#     showings = len(row)
#     return ...          # taps divided by showings

In [ ]:
# ✅ SOLUTION, task A: the estimate of one product
def estimate(row):
    """The estimate of a product from its row of 1s (taps) and 0s (no taps): taps / showings.
    None when the product was never shown."""
    if len(row) == 0:
        return None          # never shown: no estimate yet (the game writes ?)
    taps = 0
    for x in row:
        taps = taps + x      # a tap adds 1, no tap adds 0
    showings = len(row)      # one showing per dot
    return taps / showings

In [ ]:
check_estimate(estimate)

## The board: one row per product

The board has one row per product, in the order of the Mirror Shop: 🤖 Robot vacuum, 🪜 Rope ladder, 🍃 Leaf blower, 🛏️ Memory-foam mattress and 🧭 Satnav. In Python a board is a list of rows:
`race_board[0]` is the vacuum's row and `race_board[3]` the mattress's, since Python counts from 0.

On the first 25 mornings of the race season, the mirror showed the five products in turn, 5 rounds of one
showing each. Here is the board after those 25 mornings.

In [ ]:
# The first 25 mornings of the race season at Cinderella's: one row per product, in the order of the shop
race_board = [
    [0, 1, 0, 1, 1],   # 🤖 Robot vacuum
    [0, 0, 0, 0, 0],   # 🪜 Rope ladder
    [0, 0, 0, 0, 0],   # 🍃 Leaf blower
    [1, 0, 1, 0, 0],   # 🛏️ Memory-foam mattress
    [0, 0, 0, 0, 0],   # 🧭 Satnav
]
show_board(race_board)

## 🎯 Task B. The estimate of every product

Write `estimates(rows)`. It gets a board and returns a list with the estimate of every product, in the order of
the rows. Your `estimate` from task A does the work for each row. For the board above, `estimates(race_board)`
should give `[0.6, 0.0, 0.0, 0.4, 0.0]`: the p̂ column of the board.

A list grows with `append`: after `values = []` and `values.append(0.6)`, the list `values` is `[0.6]`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The loop for row in rows visits the rows of the board one by one: first the vacuum's row, then the ladder's,
# and so on.
# Your estimate from task A gives the estimate of one row: estimate(row).

In [ ]:
#@title 💡 Hint 2 for task B (double-click to open) { display-mode: "form" }
# values.append(value) puts value at the end of the list values.
# Do it inside the loop, once for each row.

In [ ]:
#@title 💡 Hint 3 for task B: the shape of the code (double-click to open) { display-mode: "form" }
#     for row in rows:
#         value = estimate(row)
#         ...                 # put value at the end of the list values

In [ ]:
# ✅ SOLUTION, task B: the estimate of every product on the board
def estimates(rows):
    """The estimate of every product on the board rows, in the order of the rows."""
    values = []
    for row in rows:
        value = estimate(row)    # the estimate of this row, from task A
        values.append(value)
    return values

In [ ]:
check_estimates(estimates)

## 🎯 Task C. The leaders

After 25 mornings the vacuum has the highest estimate. A mirror that stopped exploring now would show the
product with the highest estimate so far: in the game's words, it would exploit. Sometimes several products
share the highest estimate.

Write `leaders(values)`. It gets a list of numbers, one per product, such as the estimates of a board, and
returns the list of the rows that have the highest value, in the order of the rows. For the estimates of the 25
mornings, `[0.6, 0.0, 0.0, 0.4, 0.0]`, it gives `[0]`, the vacuum. For `[0.5, 0.0, 0.0, 0.5, 0.0]` it gives `[0, 3]`, the vacuum and the mattress, which tie.

Both loops are given:

1. The first finds the highest value: `highest` starts with the first value, and a value that is higher takes
   its place.
2. The second collects the rows that have it: `k` counts the rows from 0, so `values[k]` is the value of row k.

You write the questions that the two `if` lines ask.

💭 **Think first.** Was the vacuum ahead all along? After round 1, when every product had been shown once, which
product had the highest estimate? The cell after the check of task C shows it, with your `leaders`.

### 🎯 Your turn

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# Each if asks a question about two numbers.
# In the first loop: is value higher than highest? In Python: value > highest.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
# In the second loop: is the value of row k, values[k], equal to highest?
# In Python, == asks whether two numbers are equal. A single = gives a name a value.

In [ ]:
#@title 💡 Hint 3 for task C: the shape of the code (double-click to open) { display-mode: "form" }
#     for value in values:
#         if value > highest:
#             highest = value
#     ...
#     for k in range(len(values)):
#         if values[k] == ...:
#             best_rows.append(k)

In [ ]:
# ✅ SOLUTION, task C: the rows that share the highest value
def leaders(values):
    """The rows whose value is the highest in the list values (numbers, one per product), in the order
    of the rows."""
    # First loop: find the highest value
    highest = values[0]
    for value in values:
        if value > highest:
            highest = value
    # Second loop: collect the rows that have it
    best_rows = []
    for k in range(len(values)):
        if values[k] == highest:
            best_rows.append(k)
    return best_rows

In [ ]:
check_leaders(leaders)

## The leader, round by round

`show_rounds` draws the board after each round: the first dot of every row, then the first two dots, and so on.
Your `estimates` and `leaders` put a ⭐ on the leaders of each round.

In [ ]:
# The board of the 25 mornings after each round, with a ⭐ on the leaders that your functions find
show_rounds(race_board)

**Reading the result.** The leader changed in every round: the mattress alone after round 1 (1 tap in 1 showing), the vacuum
and the mattress tied after round 2 (0.5 each), the mattress after round 3 (0.67), a tie again after round 4 (0.5
each), and the vacuum after round 5 (0.6 against 0.4). A mirror that stopped exploring after round 1 or round 3
would have picked the mattress.

💭 **Think first.** Tomorrow is morning 26. Which product would you show Cinderella, and how sure are you? Decide,
then double-click the folded cell below.

In [ ]:
#@title 💭 One answer (double-click to open) { display-mode: "form" }
# The vacuum: after 25 mornings it has the highest estimate, 0.6, against 0.4 for the mattress.
# How sure? Not very. Each product has only 5 showings, so one tap more or less moves its estimate by 1/5,
# that is 0.2, and the leader changed in every round so far.
# The Queen's mirror of section 1 knows Cinderella's probabilities: 0.6 for the vacuum, 0.4 for the mattress,
# 0.2 for the ladder, the blower and the satnav.
# So the vacuum is the right choice. The ladder, the blower and the satnav got no tap in 5 showings: their
# estimates, 0, are 0.2 too low.
# More showings make the estimates surer, and every showing of a weaker product costs a morning: the mirror's
# dilemma of section 1, explore or exploit.

## Back to the game

That is notebook 1. From the 1s and 0s of a board, your `estimates` gives the estimate of every product, and
your `leaders` finds the products with the highest one. From notebook 2 on, the mirror's kit gives `estimate`
and `estimates` ready to use.

Back to the game: section 2, **Random**, plays a whole season with the simplest strategy of all, and notebook 2
comes after it.